In [17]:
import pandas as pd

# 1. Load the files
customers = pd.read_csv("data/customers.csv")
rfm_snapshot = pd.read_csv("data/rfm_modeling_snapshot.csv")
labels = pd.read_csv("data/churn_labels.csv")

# 2. Print exactly what Pandas sees
print("--- CUSTOMERS COLUMNS ---")
print(customers.columns.tolist())

print("\n--- RFM SNAPSHOT COLUMNS ---")
print(rfm_snapshot.columns.tolist())

print("\n--- LABELS COLUMNS ---")
print(labels.columns.tolist())

--- CUSTOMERS COLUMNS ---
['customer_id', 'signup_date', 'city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'skin_type', 'marketing_consent']

--- RFM SNAPSHOT COLUMNS ---
['customer_id', 'snapshot_date', 'city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent', 'recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d', 'category_diversity_180d', 'ticket_count_90d', 'negative_ticket_rate_90d', 'avg_resolution_hours_90d', 'days_since_signup', 'sessions_30d', 'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d', 'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d', 'last_visit_days_ago', 'churn_next_60d', 'split']

--- LABELS COLUMNS ---
['customer_id', 'snapshot_date', 'churn_next_60d', 'split']


In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load Data
customers = pd.read_csv("data/customers.csv")
rfm_snapshot = pd.read_csv("data/rfm_modeling_snapshot.csv")

# 2. CLEAN MERGE: Drop overlapping columns from the snapshot so they don't clash
duplicate_cols = ['city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent']
rfm_clean = rfm_snapshot.drop(columns=duplicate_cols + ['snapshot_date', 'split'], errors='ignore')

# Merge customers and our cleaned snapshot
df = pd.merge(customers, rfm_clean, on='customer_id')

# 3. Define Target (y) and Features (X)
y = df['churn_next_60d']
# Drop the ID, the label, and the signup date (dates confuse the machine)
X = df.drop(columns=['customer_id', 'signup_date', 'churn_next_60d'], errors='ignore')

# 4. Identify Column Types for the Pipeline dynamically
categorical_cols = ['city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'skin_type', 'marketing_consent']
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 5. Build the Preprocessing Engines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 6. Split the Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 7. Execute the Pipeline!
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Training Data Shape: {X_train_processed.shape}")
print("-" * 40)

# 8. Train the Baseline Model
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_processed, y_train)
y_pred_baseline = baseline_model.predict(X_test_processed)

# 9. Print the Scores!
print("--- Baseline Model (Logistic Regression) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_baseline):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_baseline):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_baseline):.4f}")

Training Data Shape: (1920, 50)
----------------------------------------
--- Baseline Model (Logistic Regression) ---
Accuracy:  0.7917
Precision: 0.7907
Recall:    0.7556
F1 Score:  0.7727


In [19]:
from sklearn.ensemble import RandomForestClassifier
import joblib
import json

# 1. Build the Master Pipeline (Preprocessor + Algorithm)
# class_weight='balanced' forces the model to care equally about churners and loyalists
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

# 2. Train the Pipeline on the RAW training data 
# (The pipeline handles all the scaling and encoding internally now!)
final_pipeline.fit(X_train, y_train)

# 3. Predict on the RAW test data
y_pred_rf = final_pipeline.predict(X_test)

# 4. Calculate Final Metrics
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("--- Advanced Model (Random Forest) ---")
print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1 Score:  {rf_f1:.4f}")
print("-" * 40)

# 5. Save the Deployment Artifacts!
joblib.dump(final_pipeline, 'model.pkl')
print("✅ Success! Master Pipeline saved as 'model.pkl'")

metrics = {
    "model_type": "Random Forest with scikit-learn Pipeline",
    "accuracy": round(rf_accuracy, 4),
    "precision": round(rf_precision, 4),
    "recall": round(rf_recall, 4),
    "f1_score": round(rf_f1, 4)
}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)
print("✅ Success! Metrics exported to 'metrics.json'")

--- Advanced Model (Random Forest) ---
Accuracy:  0.7875
Precision: 0.7887
Recall:    0.7467
F1 Score:  0.7671
----------------------------------------
✅ Success! Master Pipeline saved as 'model.pkl'
✅ Success! Metrics exported to 'metrics.json'
